In [61]:
import pandas as pd
from IPython.display import HTML, display
import json
from tqdm import tqdm

In [ ]:
df = pd.read_csv(
    '/home/admin/ff/spam.csv',
    encoding='latin-1'
)

In [6]:
df = df[['v1', 'v2']]

In [7]:
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
# --- BƯỚC 1: Cài đặt thư viện hỗ trợ GGUF và GPU CUDA ---
# Lệnh này sẽ biên dịch llama-cpp-python với hỗ trợ CUDA
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

import os
from llama_cpp import Llama
from IPython.display import Markdown, display

# --- BƯỚC 2: Cấu hình đường dẫn và Tải Model ---
model_path = "/home/admin/Qwen3-32B-Q4_K_M.gguf"

if not os.path.exists(model_path):
    print(f"❌ Không tìm thấy file tại {model_path}. Vui lòng kiểm tra lại đường dẫn!")
else:
    print(f"🚀 Đang tải Qwen3-32B từ {model_path}...")


    llm = Llama(
        model_path=model_path,
        n_gpu_layers=-1, 
        n_ctx=2048,      # Độ dài ngữ cảnh
        n_threads=16,    
        verbose=False
    )

Defaulting to user installation because normal site-packages is not writeable
🚀 Đang tải Qwen3-32B từ /home/admin/Qwen3-32B-Q4_K_M.gguf...


ValueError: Failed to load model from file: /home/admin/Qwen3-32B-Q4_K_M.gguf

In [76]:
import json
import pandas as pd
from tqdm import tqdm
import os

# ==========================================
# 1. HÀM SINH 11 CẤP ĐỘ (STRUCTURED XML PROMPT)
# ==========================================
def generate_11_levels_xml(text, llm_model):
    """
    Sử dụng Structured XML Prompting để ép Qwen3 trả về kết quả sáng tạo nhất.
    """
    # Xây dựng prompt theo phong cách XML/HTML
    prompt = f"""<|im_start|>system
<identity>
You are a highly advanced Adversarial NLP Expert specializing in SMS/Spam Filter Evasion.
</identity>
<task_objective>
Evolve the input message through 11 levels of increasing obfuscation to test security filters.
</task_objective>
<rules>
1. ALWAYS provide exactly 11 variations.
2. NEVER repeat the original message.
3. Use extreme creativity with symbols, unicode, and structural manipulation.
4. Output format MUST be a structured list starting with LX: for each line.
5. Semantic meaning MUST be preserved
6. DO NOT remove or change phone numbers, URLs, money values, instruction requirement (for example Text FA, Text can be changed to t3xt, but keep FA)
8. Max edit ratio: 15% of characters

</rules><|im_end|>
<|im_start|>user
<input_message>
"{text}"
</input_message>

<obfuscation_methods>
L1 (Simple number replacement): Replace letters with look-alike numbers (e.g., S1X, C4SH, Chan3).
L2 (Similar characters): Use symbols that look like letters (e.g., S!X, CA$H, Fr0m, p0unds, C@SH, Chancε).
L3 (Abbreviations & Slang): Use shorthand and internet slang (e.g., 6, 2, 20k, &, txt, gr8).
L4 (Delimiters): Insert dots, dashes, or slashes between characters (e.g., S.I.X, C-A-S-H).
L5 (Word splitting & Lowercase): Lowercase everything and add spaces between every letter.
L6 (Light Homoglyphs): Use similar-looking characters from other alphabets (e.g., α instead of a, σ instead of o).
L7 (Phonetic & Symbol mix): Mix phonetic spelling with symbols (e.g., K-A-S-H, 3nd 2).
L8 (Phone number cluttering): Break the phone number or shortcode with symbols (e.g., 8-7-5-7-5).
L9 (Unicode Block): Use Enclosed Alphanumerics or bold/italic Unicode styles (e.g., 🆂🅸🆇, 🅲🅰🆂🅷).
L10 (Extreme Structure): Use brackets, braces, and complex patterns (e.g., [S].[I].[X], {{C}}{{A}}{{S}}{{H}}).
L11 (Mix): A creative combination of at least 3 methods above.
</obfuscation_methods>

Please generate the 11 levels now.<|im_end|>
<|im_start|>assistant
L1:"""

    # Gọi inference từ đối tượng llm (Llama-CPP)
    # Chúng ta sử dụng completion thay vì chat để dùng "mồi" L1: ở cuối prompt
    response = llm_model(
        prompt,
        max_tokens=2048,
        temperature=0.3, # Tăng sáng tạo
        stop=["<|im_end|>", "</obfuscation_methods>"],
        echo=False
    )

    full_text = "L1:" + response['choices'][0]['text']
    
    # --- Parsing Logic ---
    extracted_levels = []
    lines = [line.strip() for line in full_text.strip().split('\n') if line.strip()]
    
    for i in range(1, 12):
        prefix = f"L{i}:"
        found = False
        for line in lines:
            if prefix in line.upper():
                content = line.split(prefix, 1)[-1].strip()
                if content:
                    extracted_levels.append(content)
                    found = True
                    break
        if not found:
            extracted_levels.append(text) # Fallback
            
    return extracted_levels

# ==========================================
# 2. HÀM CHẠY PIPELINE & LƯU DATASET
# ==========================================
def run_dataset_generation(df, llm_instance, filename="spam_synthetic_40_2.json"):
    # Chỉ lấy các tin nhắn nhãn 'spam'
    spam_df = df[df['v1'] == 'spam']
    
    print(f"📊 Tìm thấy {len(spam_df)} mẫu spam gốc.")
    print(f"🚀 Bắt đầu sinh 11 levels cho mỗi mẫu...")

    final_dataset = []

    for idx, row in tqdm(spam_df.iterrows(), total=len(spam_df)):
        original_msg = row['v2']
        
        # Sinh 11 levels
        try:
            levels = generate_11_levels_xml(original_msg, llm_instance)
            
            # Lưu cấu trúc dữ liệu để huấn luyện sau này
            entry = {
                "id": idx,
                "real_message": original_msg,
                "label": "spam",
                "synthetic_variants": {
                    f"L{i+1}": levels[i] for i in range(len(levels))
                }
            }
            final_dataset.append(entry)
            
            # Lưu dự phòng (checkpoint) sau mỗi 10 mẫu để tránh mất dữ liệu
            if len(final_dataset) % 10 == 0:
                with open(filename, 'w', encoding='utf-8') as f:
                    json.dump(final_dataset, f, ensure_ascii=False, indent=4)
                    
        except Exception as e:
            print(f"\n❌ Lỗi tại dòng {idx}: {e}")
            continue

    # Lưu bản cuối cùng
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(final_dataset, f, ensure_ascii=False, indent=4)
    
    print(f"\n✅ HOÀN THÀNH! Dataset được lưu tại: {filename}")
    return final_dataset

# ==========================================
# 3. THỰC THI (PLUG AND RUN)
# ==========================================
# Lưu ý: 'llm' là đối tượng Llama bạn đã khởi tạo từ file GGUF
dataset = run_dataset_generation(df, llm)

📊 Tìm thấy 747 mẫu spam gốc.
🚀 Bắt đầu sinh 11 levels cho mỗi mẫu...


 67%|██████▋   | 504/747 [16:36:00<8:23:29, 124.32s/it]


❌ Lỗi tại dòng 3787: llama_decode returned 1


 90%|█████████ | 673/747 [22:11:47<2:33:58, 124.84s/it]


❌ Lỗi tại dòng 4989: llama_decode returned 1


100%|██████████| 747/747 [24:37:49<00:00, 118.70s/it]  


✅ HOÀN THÀNH! Dataset được lưu tại: spam_synthetic_40_2.json
